# 01 · Objects & the Data Model — **Depth**

> **Pull this notebook when:** you want to know what's *actually* happening when you assign, copy,
> compare, or key a dict. Everything in the project that stores state — memory records, vocab maps,
> config — rests on this.

Depth here means the **object model**: names, references, identity, and why mutability decides what
can be a key. Predict before running; answers are hidden.

---

## 1.1 — Predict: names are references, not boxes

A variable in Python is not a box holding a value — it's a **name bound to an object**. Assignment
binds a name; it never copies. Predict all three prints.

```python
a = [1, 2, 3]
b = a              # binds b to the SAME object a names
b.append(4)
c = a[:]           # a NEW list with the same contents
c.append(5)
```

In [ ]:
a = [1, 2, 3]
b = a
b.append(4)
c = a[:]
c.append(5)
print(a)   # [1, 2, 3, 4]
print(b)   # [1, 2, 3]
print(c)   # [1, 2, 3, 4, 5]

<details>
<summary>▶ Reveal</summary>

`a` → `[1, 2, 3, 4]`, `b` → `[1, 2, 3, 4]`, `c` → `[1, 2, 3, 4, 5]`.

`b = a` bound `b` to the *same* list object, so `b.append(4)` is visible through `a` — they are two
names for one object. `c = a[:]` built a *new* list (a shallow copy), so `c.append(5)` doesn't touch
`a`. The mental model to keep: assignment moves a *name*, never the *object*. `id(a) == id(b)` but
`id(a) != id(c)`.

</details>

## 1.2 — Predict: `is` vs `==`, and a surprise

`==` asks "same value?"; `is` asks "same object?" (`id(x) == id(y)`). Predict each line — the last two
are the surprise. (`int("256")` builds the integer *at runtime* from a string, so it can't be folded
into a compile-time constant — which is what lets the surprise show.)

```python
x = [1]; y = [1]
print(x == y)            # ?
print(x is y)            # ?
a = int("256"); b = 256
print(a is b)            # ?
m = int("257"); n = 257
print(m is n)            # ?
```

In [ ]:
x = [1]; y = [1]
print(x == y) # yes
print(x is y) # no
a = int("256"); b = 256
print(a is b) # no -> yes
m = int("257"); n = 257
print(m is n) # no

<details>
<summary>▶ Reveal</summary>

`True, False, True, False`.

Lists: equal value, different objects. The integers are the surprise: CPython **pre-creates and caches
small integers** (−5 through 256) at startup and reuses them. So `int("256")` — even though built fresh
at runtime — returns *the one cached 256 object*, and `is 256` is `True`. But `257` is outside the
cache, so `int("257")` builds a genuinely new object and `is 257` is `False`.

(If you'd written the literals `a = 256; b = 256` directly in one cell, the compiler would fold both to
the same constant and even `257` could show `True` — using `int("...")` defeats that folding so the
real cache boundary is visible.)

The rule this burns in: **use `is` only for `None`, `True`, `False`** — the singletons. For everything
else, `==`. The int-caching is the one CPython internal worth knowing because it explains an `is`
result you'll otherwise find baffling.

</details>

## 1.3 — Predict, then inspect the mechanism

You met the mutable-default trap in fluency. Now look at *why* mechanically. Predict the prints, then
read what `__defaults__` reveals.

```python
def add_item(item, bucket=[]):
    bucket.append(item)
    return bucket

print(add_item(1))    # ?
print(add_item(2))    # ?
print(add_item.__defaults__)   # ? what is stored here
```

In [ ]:
def add_item(item, bucket=[]):
    bucket.append(item)
    return bucket

print(add_item(1)) # appends 1, returns [1]
print(add_item(2)) # appends 2, returns [1, 2]
print(add_item.__defaults__) # list type object with reference name bucket

<details>
<summary>▶ Reveal</summary>

`[1]`, then `[1, 2]`, then `([1, 2],)`.

The default value is evaluated **once, at definition time**, and stored on the function object itself —
you can see it in `add_item.__defaults__`, a tuple holding the *one* list. Every call that omits
`bucket` reuses that same stored list, so it accumulates across calls. `__defaults__` *is* the shared
object, sitting on the function. That's the whole bug, made visible. The fix (`bucket=None`, create
inside) means `__defaults__` holds the immutable `None`, and a fresh list is built per call.

</details>

## 1.4 — Build: shallow vs deep, by hand

A shallow copy duplicates the outer container but shares the inner objects. Write `shallow(matrix)`
(new outer list, shared inner lists) and `deep(matrix)` (fully independent), both without `import copy`.
The tests prove the difference by mutating an inner list.

(see above)

In [ ]:
def shallow(matrix):
    return matrix.copy()

def deep(matrix):
    # YOUR CODE HERE — new outer list AND new inner lists
    pass

# Tests
m = [[1, 2], [3, 4]]
s = shallow(m)
s[0].append(99)
assert m == [[1, 2, 99], [3, 4]]      # shallow: inner list shared, original changed
m2 = [[1, 2], [3, 4]]
d = deep(m2)
d[0].append(99)
assert m2 == [[1, 2], [3, 4]]         # deep: fully independent
print("1.4 passed")

<details>
<summary>▶ Why</summary>

`list(matrix)` (or `matrix[:]`) copies the *outer* list — but each element is still a reference to
the *same* inner list, so mutating `s[0]` reaches the original. `deep` rebuilds each inner list with
`row[:]`, severing the sharing. This is exactly the aliasing model from 1.1, one level down. Real deep
copies of arbitrary structures are what `copy.deepcopy` handles — you'll meet it when you persist
Mara's memory and need a snapshot that won't mutate underneath you.

</details>

## 1.5 — Predict: which copies are independent?

Five ways to "copy" a list. Predict which of `b1..b5` are independent of `a` after `a[0] = "X"`.

```python
a = ["a", "b", "c"]
b1 = a
b2 = a[:]
b3 = list(a)
b4 = a.copy()
b5 = [x for x in a]
a[0] = "X"
```

In [ ]:
a = ["a", "b", "c"]
b1 = a
b2 = a[:]
b3 = list(a)
b4 = a.copy()
b5 = [x for x in a]
a[0] = "X"
print(b1[0], b2[0], b3[0], b4[0], b5[0])

<details>
<summary>▶ Reveal</summary>

`X b b b b`.

`b1 = a` is **not a copy** — same object, sees the change. The other four (`[:]`, `list()`, `.copy()`,
comprehension) all build new outer lists, so they're independent at the top level. **When to use
which:** `a[:]` and `list(a)` are idiomatic and equivalent; `.copy()` reads most clearly as intent;
the comprehension is for when you also transform (`[f(x) for x in a]`). All four are *shallow* — for
nested data you still need the deep approach from 1.4.

</details>

## ★ Milestone 01 — An immutable value type

Synthesize the whole notebook: build `Color(r, g, b)` that behaves like a proper **value object**.

Requirements:
- equal **by value** (two `Color(1,2,3)` are equal),
- **hashable**, so it works as a dict key and dedupes in a set (equal colors must hash equal),
- a clean `repr` of the form `"Color(1, 2, 3)"`,
- it should *act* immutable — store the components in a way that signals "don't mutate me" (a tuple).

This is the data-model contract (`__eq__` + `__hash__` + `__repr__`) resting on the mutability/identity
ideas from this whole notebook. It's also the shape of any small fact you'll store in Mara's memory.

(build it below)

In [ ]:
class Color:
    def __init__(self, r, g, b):
        # YOUR CODE HERE — store components (as a tuple), expose r, g, b
        pass

    def __eq__(self, other):
        # YOUR CODE HERE
        pass

    def __hash__(self):
        # YOUR CODE HERE — equal colors must hash equal
        pass

    def __repr__(self):
        # YOUR CODE HERE
        pass

# Tests
assert Color(1, 2, 3) == Color(1, 2, 3)
assert Color(1, 2, 3) != Color(9, 9, 9)
assert repr(Color(1, 2, 3)) == "Color(1, 2, 3)"
# hashable: dedupes in a set, usable as a dict key
palette = {Color(0, 0, 0), Color(0, 0, 0), Color(255, 255, 255)}
assert len(palette) == 2
named = {Color(255, 0, 0): "red"}
assert named[Color(255, 0, 0)] == "red"     # lookup by an equal-but-different object
print("Milestone 01 passed")

<details>
<summary>▶ The contract</summary>

Two methods make the value type work, and they must agree: `__eq__` defines when two colors are the
same (same components), and `__hash__` must return equal hashes for equal colors — done by hashing the
underlying tuple `(r, g, b)`. Because tuples are immutable and hashable, hashing them is free and safe.
That agreement is the **hash/eq contract**: if `a == b` then `hash(a) == hash(b)`, or sets and dicts
break. You'll see the dark side of breaking it in notebook 05.

</details>